In [0]:
# ============================================================
# 02_SILVER_CLEANING
# Bronze → Clean, fix types, handle nulls → Silver Delta Table
# ============================================================

BRONZE_TABLE = "workspace.default.bronze_loan_applications"
SILVER_TABLE = "workspace.default.silver_loan_applications"

# Read from Bronze
df_bronze = spark.table(BRONZE_TABLE)
print("✅ Rows from Bronze:", df_bronze.count())
print("✅ Columns:", len(df_bronze.columns))

✅ Rows from Bronze: 307511
✅ Columns: 125


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count nulls per column
null_counts = df_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

# Show only columns that have nulls
null_df = null_counts.toPandas().T
null_df.columns = ["null_count"]
null_df = null_df[null_df["null_count"] > 0].sort_values("null_count", ascending=False)
print("Columns with nulls:")
print(null_df.head(20))

Columns with nulls:
                          null_count
COMMONAREA_MEDI               214865
COMMONAREA_AVG                214865
COMMONAREA_MODE               214865
NONLIVINGAPARTMENTS_MEDI      213514
NONLIVINGAPARTMENTS_MODE      213514
NONLIVINGAPARTMENTS_AVG       213514
FONDKAPREMONT_MODE            210295
LIVINGAPARTMENTS_MODE         210199
LIVINGAPARTMENTS_MEDI         210199
LIVINGAPARTMENTS_AVG          210199
FLOORSMIN_MODE                208642
FLOORSMIN_MEDI                208642
FLOORSMIN_AVG                 208642
YEARS_BUILD_MODE              204488
YEARS_BUILD_MEDI              204488
YEARS_BUILD_AVG               204488
OWN_CAR_AGE                   202929
LANDAREA_AVG                  182590
LANDAREA_MEDI                 182590
LANDAREA_MODE                 182590


In [0]:
# Drop bronze metadata columns
meta_cols = ["_ingested_at", "_source_file", "_layer"]
df = df_bronze.drop(*meta_cols)

# Drop columns with more than 40% nulls
threshold = 0.4
total_rows = df.count()

high_null_cols = [
    c for c in df.columns
    if df.filter(col(c).isNull()).count() / total_rows > threshold
]

print(f"Dropping {len(high_null_cols)} high-null columns:", high_null_cols)
df = df.drop(*high_null_cols)
print(f"✅ Remaining columns: {len(df.columns)}")

Dropping 49 high-null columns: ['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSMATER

In [0]:
from pyspark.sql.functions import when, trim

# Clean binary string columns → proper values
df = df \
    .withColumn("FLAG_OWN_CAR", when(col("FLAG_OWN_CAR") == "Y", 1).otherwise(0)) \
    .withColumn("FLAG_OWN_REALTY", when(col("FLAG_OWN_REALTY") == "Y", 1).otherwise(0)) \
    .withColumn("CODE_GENDER", when(col("CODE_GENDER") == "M", 1)
                                .when(col("CODE_GENDER") == "F", 0)
                                .otherwise(None))

# Trim whitespace from all string columns
str_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
for c in str_cols:
    df = df.withColumn(c, trim(col(c)))

print("✅ Data types cleaned")

✅ Data types cleaned


In [0]:
# Debug: check schema after Cell 4
from pyspark.sql.types import DoubleType, IntegerType, FloatType, LongType, StringType

print("=== NUMERIC COLS ===")
num_cols = [f.name for f in df.schema.fields 
            if isinstance(f.dataType, (DoubleType, IntegerType, FloatType, LongType))]
print(num_cols)

print("\n=== STRING COLS ===")
str_cols = [f.name for f in df.schema.fields 
            if isinstance(f.dataType, StringType)]
print(str_cols)

print("\n=== FULL SCHEMA ===")
df.printSchema()

=== NUMERIC COLS ===
['SK_ID_CURR', 'TARGET', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_D

In [0]:
# Cell 5 — Fill Nulls (simple & reliable)

# Numeric cols (exclude TARGET and ID)
num_cols = [f.name for f in df.schema.fields 
            if str(f.dataType) in ["DoubleType()", "IntegerType()"]
            and f.name not in ["TARGET", "SK_ID_CURR"]]

# Fill numeric nulls with 0 (safe default for this dataset)
df = df.fillna(0, subset=num_cols)

# Fill string nulls with UNKNOWN
str_cols = [f.name for f in df.schema.fields 
            if str(f.dataType) == "StringType()"]
df = df.fillna("UNKNOWN", subset=str_cols)

# Verify no nulls remain
from pyspark.sql.functions import col, sum as spark_sum
null_check = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_totals = null_check.toPandas().T
null_totals.columns = ["null_count"]
remaining = null_totals[null_totals["null_count"] > 0]
print("Remaining nulls:", len(remaining))
print(remaining)
print("✅ Nulls filled successfully")

Remaining nulls: 0
Empty DataFrame
Columns: [null_count]
Index: []
✅ Nulls filled successfully


In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add silver metadata
df_silver = df \
    .withColumn("_processed_at", current_timestamp()) \
    .withColumn("_layer", lit("silver"))

# Write Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"✅ Silver table written: {SILVER_TABLE}")

✅ Silver table written: workspace.default.silver_loan_applications


In [0]:
df_check = spark.table(SILVER_TABLE)
print("✅ Silver rows:", df_check.count())
print("✅ Silver columns:", len(df_check.columns))

# Check target distribution
print("\nTarget Distribution (0=No Default, 1=Default):")
df_check.groupBy("TARGET").count().orderBy("TARGET").show()

✅ Silver rows: 307511
✅ Silver columns: 75

Target Distribution (0=No Default, 1=Default):
+------+------+
|TARGET| count|
+------+------+
|     0|282686|
|     1| 24825|
+------+------+

